# Emergence of functional patterns
This notebook was used for the creation of figures evaluating the LOV dataset.

In [ ]:
# read LOV dataset exported from SelectZyme
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from matplotlib.colors import BoundaryNorm

df = pd.read_csv("lov_10_15.tsv", sep="\t")
print(df.shape)
df.head()

# KNN agreement

In [ ]:
def _ensure_xy_numeric(df):
    df = df.copy()
    for c in ["x", "y"]:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.replace(",", ".", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def knn_label_consistency_scores(df, label_column, k=10, n_perm=200, dropna=True, seed=0):
    """
    Returns:
      real_score: mean kNN agreement
      null_mean, mad: mean absolute deviation (permutation baseline)
    """
    rng = np.random.default_rng(seed)
    df2 = _ensure_xy_numeric(df)

    cols = ["x", "y", label_column]
    if dropna:
        df2 = df2.dropna(subset=cols)

    coords = df2[["x", "y"]].to_numpy()
    labels = df2[label_column].astype("category")
    y = labels.cat.codes.to_numpy()  # int labels; -1 means NaN (shouldn’t happen after dropna)

    # Fit kNN once
    knn = NearestNeighbors(n_neighbors=k).fit(coords)
    neigh_idx = knn.kneighbors(return_distance=False)

    # Real score
    real = np.mean([(y[neigh] == y[i]).mean() for i, neigh in enumerate(neigh_idx)])

    # Permutation null (preserves class sizes, breaks spatial structure)
    null_scores = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        s = np.mean([(y_perm[neigh] == y_perm[i]).mean() for i, neigh in enumerate(neigh_idx)])
        null_scores.append(s)

    null_mean = float(np.mean(null_scores))
    mad = float(np.mean(np.abs(np.array(null_scores) - null_mean))) if len(null_scores) > 1 else np.nan

    return real, null_mean, mad

def plot_knn_consistency_summary(df, columns, k=10, n_perm=200, seed=0, title=None):
    rows = []
    for col in columns:
        real, null_mean, mad = knn_label_consistency_scores(
            df, col, k=k, n_perm=n_perm, seed=seed
        )
        rows.append({
            "column": col,
            "knn_real": real,
            "knn_null_mean": null_mean,
            "mad": mad
        })

    res = pd.DataFrame(rows).sort_values("knn_real", ascending=True)  # sort so best are at top if we invert later

    # Plot: real agreement and null mean as a reference
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(columns))))

    y_pos = np.arange(len(res))
    ax.barh(y_pos, res["knn_real"], xerr=res["mad"], alpha=0.9, label="kNN agreement")

    # Null mean marker
    ax.scatter(res["knn_null_mean"], y_pos, marker="|", s=250, label="Permutation mean")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(res["column"])
    ax.set_xlabel(f"kNN agreement (k={k})")
    ax.grid(True, axis="x", alpha=0.3)
    ax.legend(loc="lower right", frameon=False)

    plt.show()
    return res


In [ ]:
# Define columns to evaluate
cols_main = [
    "Accession", "Domain Structure", "Functional Cluster", "Primary Effector",
    "Primary Effector Gene Ontology", "Order", "Class", "cluster", 
]
cols_negative = ["Accession", "GenBank Accession", "Species", "sequence"]

res_main = plot_knn_consistency_summary(df, cols_main, k=16, n_perm=200, seed=1)
# res_neg  = plot_knn_consistency_summary(df, cols_negative, k=16, n_perm=200, seed=1)

In [ ]:
def plot_knn_vs_k_multi(
    df,
    cols_main,
    ks=(1,2,4,8,16,32,64,128),
    n_perm=200,
    seed=42,
):
    """
    Single figure:
      - One 'real' curve per annotation in cols_main
      - One 'null mean' curve per annotation (all same style/color)
      - Only ONE legend entry for all null curves
    """
    plt.figure(figsize=(9,5))

    # Plot REAL lines (each gets its own legend entry)
    real_lines = []
    for col in cols_main:
        real_scores = []
        null_means  = []
        for k in ks:
            real, null_mean, mad = knn_label_consistency_scores(
                df, col, k=k, n_perm=n_perm, seed=seed
            )
            real_scores.append(real)
            null_means.append(null_mean)

        # Real line
        line_real, = plt.plot(ks, real_scores, marker="o", linewidth=2, label=f"{col}")
        real_lines.append(line_real)

        # Null mean line (same style for all; only label the first one)
        null_label = "null mean (all annotations)" if col == cols_main[0] else "_nolegend_"
        plt.plot(ks, null_means, marker="o", linestyle="--", linewidth=1.5,
                 color="gray", alpha=0.75, label=null_label)

    plt.xlabel("k")
    plt.ylabel("kNN agreement")
    plt.title(f"kNN agreement vs k")
    plt.grid(True, alpha=0.3)
    plt.xscale("log", base=2)  # nice for ks like 1,2,4,8,...
    plt.xticks(ks, [str(k) for k in ks])
    plt.legend()
    plt.show()


In [ ]:
# cols_main = ["Primary Effector", "Functional Cluster", "Order", "Accession"]
cols_main = ["Accession", "Domain Structure", "Functional Cluster", "Primary Effector", "Primary Effector Gene Ontology", "Order", "Class", "cluster"]

plot_knn_vs_k_multi(
    df,
    cols_main=cols_main,
    ks=(1,2,4,8,16,32,64,128),
    n_perm=200,
    seed=42
)

In [ ]:
def _ensure_xy_numeric(df):
    df = df.copy()
    for c in ["x", "y"]:
        if df[c].dtype == "object":
            df[c] = df[c].astype(str).str.replace(",", ".", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def _collapse_top_categories(
    s: pd.Series,
    top_n: int = 12,
    other_label: str = "Other",
    min_keep_count: int = 2,
    na_label: str = "NA",
):
    """
    Collapse long-tail categories to `other_label`.

    - Keeps up to `top_n` categories with count >= `min_keep_count` (ranked by frequency).
    - If nothing meets `min_keep_count` (e.g. all unique IDs like Accession), collapses *everything* to `other_label`.
    """
    s2 = s.astype(str).fillna(na_label)
    vc = s2.value_counts(dropna=False)

    keep = vc[vc >= min_keep_count].head(top_n).index.tolist()
    if len(keep) == 0:
        return pd.Categorical([other_label] * len(s2), categories=[other_label], ordered=False)

    s_collapsed = s2.where(s2.isin(keep), other_label)
    categories = keep + ([other_label] if (s_collapsed == other_label).any() else [])
    return pd.Categorical(s_collapsed, categories=categories, ordered=False)

def plot_umap_by_annotations(
    df,
    cols_main,
    x_col: str = "x",
    y_col: str = "y",
    ncols: int = 2,
    point_size: float = 6,
    alpha: float = 0.75,
    top_n: int = 12,
    other_label: str = "Other",
    min_keep_count: int = 2,
    dropna_xy: bool = True,
):
    df2 = _ensure_xy_numeric(df)
    if dropna_xy:
        df2 = df2.dropna(subset=[x_col, y_col]).copy()

    n_panels = len(cols_main)
    nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(8 * ncols, 6 * nrows),
        constrained_layout=True,
    )
    axes = np.array(axes).reshape(-1)

    x = df2[x_col].to_numpy()
    y = df2[y_col].to_numpy()

    for ax_i, col in enumerate(cols_main):
        ax = axes[ax_i]
        if col not in df2.columns:
            ax.set_title(f"{col} (missing)")
            ax.axis("off")
            continue

        s = df2[col]
        unique_n = s.nunique(dropna=False)

        # Collapse high-cardinality annotations (or effectively-unique ID columns).
        if unique_n > (top_n + 1):
            s_plot = _collapse_top_categories(
                s,
                top_n=top_n,
                other_label=other_label,
                min_keep_count=min_keep_count,
            )
            labels = list(s_plot.categories)
        else:
            # For low-cardinality: order legend by frequency (most abundant first).
            s2 = s.astype(str).fillna("NA")
            vc = s2.value_counts(dropna=False)
            labels = vc.index.tolist()
            s_plot = pd.Categorical(s2, categories=labels, ordered=False)

        # Always put 'Other' last (both legend and mapping).
        if other_label in labels:
            labels = [lab for lab in labels if lab != other_label] + [other_label]
            s_plot = pd.Categorical(pd.Series(s_plot), categories=labels, ordered=False)

        # Integer codes 0..K-1 aligned to `labels`
        codes = pd.Series(s_plot).map({lab: i for i, lab in enumerate(labels)}).to_numpy()
        if np.any(pd.isna(codes)):
            # Shouldn't happen, but guard just in case
            codes = np.nan_to_num(codes, nan=0).astype(int)
        else:
            codes = codes.astype(int)

        K = len(labels)
        cmap = plt.get_cmap("tab20", K)  # discrete LUT
        norm = BoundaryNorm(np.arange(-0.5, K + 0.5, 1), K)

        ax.scatter(
            x,
            y,
            c=codes,
            cmap=cmap,
            norm=norm,
            s=point_size,
            alpha=alpha,
            linewidths=0,
        )

        shown_top = K - 1 if other_label in labels else K
        ax.set_title(f"{col} (only top {shown_top} shown)", fontsize=14)
        ax.set_xlabel("UMAP-1")
        ax.set_ylabel("UMAP-2")
        ax.grid(True, alpha=0.2)

        # Legend colors must match the discrete LUT exactly
        handles = []
        for i, lab in enumerate(labels):
            color = cmap.colors[i] if hasattr(cmap, "colors") else cmap(i)
            handles.append(
                plt.Line2D(
                    [0],
                    [0],
                    marker="o",
                    color="none",
                    markerfacecolor=color,
                    markersize=6,
                    alpha=alpha,
                    label=lab,
                )
            )
        ax.legend(
            handles=handles,
            title=col,
            loc="upper right",
            frameon=False,
            fontsize=10,
            title_fontsize=12,
        )

    for j in range(n_panels, len(axes)):
        axes[j].axis("off")

    plt.show()

In [ ]:
cols_main = ["Primary Effector", "Functional Cluster", "Order", "Accession"]
plot_umap_by_annotations(
    df,
    cols_main=cols_main,
    ncols=2,
    point_size=6,
    alpha=0.65,
    top_n=10,
    other_label="Other",
    min_keep_count=2,
 )